# Plain SAC baseline

Soft Actor-Critic on CarRacing-v3 pixels with the same environment fixes as V-CARA (brake mapping, time-aware input) but none of the V-CARA components: no counterfactual evaluation, no risk or progress heads, no finish head, no elite replay. Use it as the reference for what the extra machinery adds.

Note: the `fixed_horizon` here is 1 (one-step targets).

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from vcara import make_cfg, train

VARIANT = "sac"
SEEDS = [1, 2, 3, 4]          # training seeds to load (set to the seeds you ran)
OUT = ROOT / "outputs"
RUN_TRAINING = False          # True to train here (slow, GPU recommended)
RECORD_VIDEO = False          # True to render one evaluation episode into videos/

## Configuration

In [ ]:
cfg = make_cfg(VARIANT)
keys = ["use_cf", "use_gate", "use_risk", "use_progress", "use_finish", "use_elite_replay",
        "fixed_horizon", "time_aware", "stall_mode", "total_frames"]
pd.Series({k: getattr(cfg, k) for k in keys}, name=VARIANT)

## Training (optional)

In [ ]:
if RUN_TRAINING:
    for s in SEEDS:
        c = make_cfg(VARIANT)
        train(c, name=VARIANT, seed=s, out_dir=str(OUT), verbose=True)
else:
    print("Training skipped. Set RUN_TRAINING = True to train here, or run:")
    print(f"  experiments/run_multiseed.sh {VARIANT} 1000000 {' '.join(map(str, SEEDS))}")

## Training logs

In [ ]:
rows, logs = [], {}
for s in SEEDS:
    p = OUT / f"{VARIANT}_seed{s}.json"
    if not p.exists():
        print("missing", p.name); continue
    logs[s] = json.loads(p.read_text())
    for e in logs[s]["evals"]:
        rows.append(dict(seed=s, frames=e["frames"], final=bool(e.get("final", False)),
                         mean_return=e["mean_return"], std_return=e["std_return"],
                         mean_coverage=e["mean_coverage"], off_road_frac=e["off_road_frac"],
                         completion=e["completion"], mean_speed=e["mean_speed"], t90=e["t90"]))
df = pd.DataFrame(rows)
print(f"{len(logs)} seed log(s) loaded, {len(df)} evaluation rows")
df.tail()

## Return over training

In [ ]:
ev = df[~df["final"]]
fig, ax = plt.subplots(figsize=(8, 4.5))
for s, g in ev.groupby("seed"):
    ax.plot(g["frames"] / 1e3, g["mean_return"], alpha=0.5, lw=1, label=f"seed {s}")
agg = ev.groupby("frames")["mean_return"].agg(["mean", "std"]).reset_index()
ax.plot(agg["frames"] / 1e3, agg["mean"], "k-", lw=2, label="across-seed mean")
ax.fill_between(agg["frames"] / 1e3, agg["mean"] - agg["std"], agg["mean"] + agg["std"], color="k", alpha=0.1)
ax.set_xlabel("environment frames (thousands)"); ax.set_ylabel("evaluation return")
ax.set_title(f"{VARIANT}: return over training"); ax.legend(fontsize=8); fig.tight_layout()

## Coverage and off-road

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s, g in ev.groupby("seed"):
    axes[0].plot(g["frames"] / 1e3, g["mean_coverage"], lw=1, label=f"seed {s}")
    axes[1].plot(g["frames"] / 1e3, g["off_road_frac"], lw=1, label=f"seed {s}")
axes[0].set(title="mean coverage", xlabel="frames (k)", ylabel="coverage")
axes[1].set(title="off-road fraction", xlabel="frames (k)", ylabel="fraction")
for a in axes: a.grid(alpha=0.3)
axes[0].legend(fontsize=8); fig.tight_layout()

## Auxiliary diagnostics

In [ ]:
diag = pd.DataFrame([dict(seed=s, **d) for s, l in logs.items() for d in l["diag"]])
wanted = [c for c in ["changed", "G", "cf_u", "loss_q", "q_gap", "alpha", "risk_loss",
                      "fin_bacc", "elite_frac", "near_frac"] if c in diag.columns]
print("diagnostic columns available:", wanted or "none (this variant has few auxiliary heads)")
if wanted:
    fig, axes = plt.subplots(len(wanted), 1, figsize=(9, 2.2 * len(wanted)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, wanted):
        for s, g in diag.groupby("seed"):
            ax.plot(g["frames"] / 1e3, g[c], lw=1)
        ax.set_ylabel(c)
    axes[-1].set_xlabel("frames (k)")
    fig.tight_layout()

## Held-out evaluation across seeds

Run `scripts/evaluate.py` first; the table shows across-seed mean and sample SD on the final evaluation seeds.

In [ ]:
p = ROOT / "results" / "metrics" / f"eval_{VARIANT}.json"
if p.exists():
    res = json.loads(p.read_text())
    print(f"held-out evaluation over {res['n_seeds']} training seeds, eval seeds {res['eval_seeds'][0]}-{res['eval_seeds'][-1]}")
    pd.DataFrame(res["aggregate"]).T[["mean", "sd_across_seeds", "n"]]
else:
    print("No evaluation file yet. Run:  python scripts/evaluate.py --variants", VARIANT, "--seeds", *SEEDS)

## Video (optional)

In [ ]:
if RECORD_VIDEO:
    import torch
    from vcara import VCARAAgent, evaluate
    from IPython.display import Video
    import subprocess
    ck = OUT / "ckpts" / f"{VARIANT}_seed{SEEDS[0]}" / "final.pt"
    agent = VCARAAgent(cfg, "cuda" if torch.cuda.is_available() else "cpu")
    agent.load_policy(str(ck))
    stats, frames = evaluate(agent, cfg, [cfg.final_eval_seeds[0]], record=True)
    print({k: round(v, 3) for k, v in stats.items() if isinstance(v, float)})
    subprocess.run(["python", str(ROOT / "scripts" / "record_videos.py"), "--ckpt", str(ck),
                    "--variant", VARIANT, "--seeds", str(cfg.final_eval_seeds[0]),
                    "--out", str(ROOT / "videos" / VARIANT)], check=True)
else:
    print("Video recording skipped. Set RECORD_VIDEO = True.")

## Caveats

- Return SD in the logs is within-evaluation variation, not across-seed.
- Compare variants on the same seeds and the same final evaluation seeds only.
- Four seeds is a small sample; treat differences as provisional.